# Leçon 12 - Réduction de l’historique de chat avec le carnet de notes de l’agent

Ce notebook montre comment gérer le contexte dans les longues conversations en utilisant le Microsoft Agent Framework. À mesure que les conversations s’allongent, le nombre de tokens augmente — finissant par dépasser la fenêtre de contexte du modèle. Nous abordons cela avec un **modèle de résumé de contexte** et un **carnet de notes de l’agent** pour une mémoire persistante.

## Ce que vous apprendrez :
1. **Pourquoi la gestion du contexte est importante** : comprendre les limites de tokens et les fenêtres de contexte
2. **Agents conscients du contexte** : construire des agents qui gèrent leur propre contexte de conversation
3. **Modèle de résumé de contexte** : utiliser des outils pour condenser l’historique de la conversation
4. **Carnet de notes de l’agent** : mémoire persistante qui survit à la réduction du contexte

## Prérequis :
- Configuration Azure OpenAI avec les variables d’environnement définies
- Compréhension des concepts de base des agents des leçons précédentes


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Pourquoi la gestion du contexte est importante

Chaque LLM a une **fenêtre de contexte** finie — le nombre maximum de tokens qu'il peut traiter dans une seule requête. Au fur et à mesure qu'une conversation à plusieurs tours progresse :

- Le **nombre de tokens croît linéairement** avec chaque message utilisateur et réponse de l'assistant.
- Les **tokens du prompt dominent le coût** car tout l'historique est renvoyé à chaque tour.
- Finalement, la conversation **dépasse la fenêtre de contexte** et le modèle tronque ou génère une erreur.

### Stratégies pour gérer le contexte

| Stratégie | Fonctionnement | Compromis |
|---|---|---|
| **Troncature** | Supprimer les messages les plus anciens | Perte de contexte initial |
| **Résumé** | Condenser les anciens messages en un résumé | Quelques détails perdus, mais les points clés conservés |
| **Tableau de notes / Mémoire externe** | Stocker les faits clés en dehors de la conversation | Nécessite des appels d'outils, mais survit à toute réduction |

Dans ce notebook, nous combinons le **résumé** avec un **outil tableau de notes** afin que l'agent puisse maintenir la continuité même lorsque l'historique de la conversation est condensé.


## Créer un agent contextuel


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulation d’une longue conversation

Parcourons une conversation à plusieurs reprises pour voir comment le contexte s’accumule. L’agent doit retenir les détails clés (préférences, budget, dates de voyage) au fil des échanges et démontrer une continuité.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Notez comment l'agent conserve le contexte des tours précédents — il connaît le Japon, le sushi, les temples, la photographie, le budget de 3000 $, le voyage en solo et la période d'avril. Dans une courte conversation, cela fonctionne bien, mais à mesure que la conversation s'allonge, renvoyer tout l'historique devient coûteux.

Continuons la conversation avec plus de tours pour voir l'accumulation du contexte :


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Modèle de Résumé du Contexte

Au fur et à mesure que la conversation évolue, nous pouvons utiliser un **outil de résumé** pour condenser le contexte accumulé en un format compact. L'agent utilise cet outil pour enregistrer les préférences clés afin que même si les messages plus anciens sont supprimés, l'information essentielle soit préservée.

Ce modèle est la base pour une réduction de l'historique plus sophistiquée :
1. L'agent identifie les faits clés de la conversation
2. Il utilise l'outil de résumé pour les conserver
3. Les messages anciens peuvent être supprimés en toute sécurité car le résumé capture l'essentiel

Ci-dessous, nous définissons un outil `summarize_preferences` que l'agent peut utiliser pour enregistrer un résumé compact de ce qu'il a appris.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Résumé

Dans cette leçon, vous avez appris à gérer le contexte dans les conversations d'agents à longue durée avec Microsoft Agent Framework :

### Concepts clés
- **Les fenêtres de contexte sont limitées** — chaque jeton dans l'historique de conversation coûte de l'argent et compte dans la limite.
- **Les outils de synthèse** permettent à l'agent de condenser le contexte accumulé en résumés compacts, réduisant l'utilisation des jetons tout en conservant les informations essentielles.
- **Les blocs-notes d’agent** fournissent une mémoire externe persistante qui survit à toute réduction de conversation.

### Ce que vous avez construit
- Un **agent conscient du contexte** qui maintient la continuité sur des conversations à plusieurs tours
- Un **outil de synthèse** (`summarize_preferences`) qui enregistre les détails clés de l'utilisateur dans un format compact
- Une **conversation à plusieurs tours** démontrant la rétention du contexte et la gestion des changements

### Applications dans le monde réel
- **Bots de service client** : Mémoriser les préférences pendant de longues sessions de support
- **Assistants personnels** : Suivre des projets en cours sans réexpliquer le contexte
- **Tuteurs éducatifs** : Maintenir le progrès des élèves à travers de nombreuses interactions

### Étapes suivantes
- Implémenter un outil complet de bloc-notes avec persistance basée sur des fichiers
- Ajouter une troncature automatique de l’historique après la synthèse
- Combiner avec des bases de données vectorielles pour la recherche de mémoire sémantique
- Construire des agents capables de reprendre les conversations plusieurs jours plus tard avec le contexte complet


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
